# Round 17: parameter diversity (feature_fraction) + delay summary - S6E10

Two bounded experiments, one change each, on the strongest measured scheme (Exp 36):

- **Exp 42: column-subsampling diversity.** Seed averaging won twice in this campaign (rounds
  10 and 12); feature_fraction is the other classic stochastic axis. Exp 36 exactly, one
  change: feature_fraction 0.9 -> 0.8 across the six fits.
- **Exp 43: delay summary.** Delays are the strongest domain driver of airline satisfaction
  that the scheme carries only as two raw columns. delay_total (Departure + Arrival,
  NaN-preserving) is added as one RAW feature (not counted, not target-encoded); counts/TE
  stay on the original 21 columns. A distinct family from the round-16 rating aggregates,
  which measured slightly negative.

No new forest config this round: the measured forest averaging gain (+0.000104, round 16)
sits an order below the measured forest-booster gap (-0.00226), so further forest variants
wait for a measured case that earns them; the standing RF preference continues to be honored
with measured tests when one does.

Guard: refit fires only on a WORKING SIGNAL win vs the Exp 36 bar. 0.960 status, two facts:
the preferred candidate is Exp 36 at 0.959996 (strictly below 0.960); the campaign's highest
measured number (0.960018, round-15 blend scan) is an exploratory adaptive same-OOF scan
output, not a candidate score.


In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CROSS_PAIRS = [('Class', 'Type of Travel'), ('Customer Type', 'Class')]
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, round 9
E36_FOLDS_R12 = [0.96016, 0.95887, 0.96067, 0.96009, 0.96019]       # the bar (seed-42 partition)


In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def cross_name(pair):
    return ' x '.join(pair)

CROSS_COLS = [cross_name(p) for p in CROSS_PAIRS]

def add_crosses(df):
    out = df.copy()
    for pair in CROSS_PAIRS:
        out[cross_name(pair)] = out[pair[0]].astype(str) + ' x ' + out[pair[1]].astype(str)
    return out

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False,
             target_encode=False, te_m=TE_SMOOTH, te_cols=None, te_drop=(),
             outer_seed=RANDOM_STATE, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)  # round 14: partition passed in
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 audit fix)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            cols = te_cols if te_cols is not None else TE_COLS_ALL
            te_fit = te_crossfit(X_fit, y_fit, cols, m=te_m)
            te_es = te_apply(X_fit, y_fit, X_es, cols, m=te_m)
            te_va = te_apply(X_tr, y_tr, X_va, cols, m=te_m)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
            if te_drop:  # raw crossed strings never enter the model - only their encodings
                X_fit = X_fit.drop(columns=list(te_drop))
                X_es = X_es.drop(columns=list(te_drop))
                X_va = X_va.drop(columns=list(te_drop))
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label, outer_seed=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; crossed columns: {CROSS_COLS}')


def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label


## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every seed-42-partition round.


In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

## Exp 42: Exp 36 scheme with feature_fraction 0.8

Exp 36 exactly (counts + all-21 TE + stochastic, 3 seeds x {63, 127} leaves), one change:
feature_fraction 0.9 -> 0.8. Seed averaging produced this campaign's two measured wins;
column subsampling is the other stochastic diversity axis, tested as a single parameter step.


In [ ]:
STOCH_FF8 = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.8)
print(f'Exp 42: counts + all-{len(TE_COLS_ALL)} TE + stochastic with feature_fraction 0.8 (Exp 36 used 0.9), 3 seeds x {{63, 127}} leaves')
LEAF_CONFIGS = [63, 127]
oof_runs42, iters42 = {}, {}
for leaves in LEAF_CONFIGS:
    for seed in [42, 43, 44]:
        o, a, it = oof_lgbm(features, target, f'E42-l{leaves}-s{seed}', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, **STOCH_FF8)
        oof_runs42[(leaves, seed)] = o
        iters42[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof42 = np.mean(list(oof_runs42.values()), axis=0)
aucs42 = fold_aucs(oof42, target)
print(f'Exp 42 (averaged) fold AUCs: {[round(a, 5) for a in aucs42]}')
print(f'Exp 42 mean OOF AUC: {np.mean(aucs42):.6f} +/- {np.std(aucs42):.5f} | pooled: {roc_auc_score(target, oof42):.6f}')


## Exp 43: Exp 36 scheme + delay_total

The two delay columns enter the scheme raw. delay_total = Departure + Arrival Delay is the
trip-level disruption summary, added as one RAW feature with NaN preserved (a NaN in either
leg yields a NaN total; LightGBM handles NaN natively, as it already does for the raw delay
columns). Counts and TE remain on the original 21 columns only - the new feature is neither
counted nor target-encoded. Delay columns are derived by name and printed.


In [ ]:
DELAY_COLS = [c for c in features.columns if 'Delay' in c]
assert len(DELAY_COLS) == 2, f'unexpected delay columns: {DELAY_COLS}'
print(f'2 delay columns identified by name: {DELAY_COLS}')

def add_delay_total(df):
    out = df.copy()
    out['delay_total'] = out[DELAY_COLS].sum(axis=1, skipna=False)  # NaN preserved
    return out

features43 = add_delay_total(features)
print(f'Exp 43 frame: {features43.shape[1]} columns ({features.shape[1]} raw + 1 delay aggregate); '
      f'counts/TE still on the original {len(TE_COLS_ALL)}; '
      f'delay_total NaNs: {features43["delay_total"].isna().sum()}')

print(f'Exp 43: counts + all-{len(TE_COLS_ALL)} TE + raw delay_total + stochastic, 3 seeds x {{63, 127}} leaves')
oof_runs43, iters43 = {}, {}
for leaves in LEAF_CONFIGS:
    for seed in [42, 43, 44]:
        o, a, it = oof_lgbm(features43, target, f'E43-l{leaves}-s{seed}', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, **STOCH)
        oof_runs43[(leaves, seed)] = o
        iters43[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof43 = np.mean(list(oof_runs43.values()), axis=0)
aucs43 = fold_aucs(oof43, target)
print(f'Exp 43 (averaged) fold AUCs: {[round(a, 5) for a in aucs43]}')
print(f'Exp 43 mean OOF AUC: {np.mean(aucs43):.6f} +/- {np.std(aucs43):.5f} | pooled: {roc_auc_score(target, oof43):.6f}')


## Verdicts and refit guard

Each candidate pairs against the in-run incumbent, the Exp 29 folds, and the Exp 36 bar.
Refit fires only on a WORKING SIGNAL better vs the bar; on a tie or loss, Exp 36 and
submission_round12.csv stand. Bounded to the tested configurations.


In [ ]:
results = {}
for tag, aucs, desc in [('42', aucs42, 'Exp 42: E36 scheme with feature_fraction 0.8'),
                        ('43', aucs43, 'Exp 43: E36 scheme + raw delay_total')]:
    print(desc)
    m_, s_, lab = paired_verdict(aucs, INC_FOLDS)
    print(f'  vs incumbent (in-run):          mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_, s_, lab = paired_verdict(aucs, E29_FOLDS_R9)
    print(f'  vs Exp 29 folds (round 9):      mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_bar, s_bar, lab_bar = paired_verdict(aucs, E36_FOLDS_R12)
    print(f'  vs Exp 36 folds (round 12 bar): mean delta {m_bar:+.5f} (spread {s_bar:.5f}) - {lab_bar}')
    results[tag] = (m_bar, lab_bar, float(np.mean(aucs)))

winners = [k for k, (m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: results[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 36 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 36 bar under the paired band. Exp 36 remains the '
          'preferred candidate; submission_round12.csv is untouched.')


## Conditional refit (only if the guard fired)

Either branch: round-12 refit semantics on its own frame (full-train counts + cross-fit TE on
the original 21 columns, the experiment's raw addition carried raw, 3 seeds x {63, 127}
leaves, n_estimators basis from the in-run seed-42 fold-mean best_iterations, logged).


In [ ]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win == '42':
        feats_full, feats_test, stoch = features, test_features, STOCH_FF8
        tag = 'Exp 42: E36 scheme with feature_fraction 0.8'
    else:  # win == '43'
        feats_full, feats_test, stoch = add_delay_total(features), add_delay_total(test_features), STOCH
        tag = 'Exp 43: E36 scheme + raw delay_total'
    iters_map = iters42 if win == '42' else iters43
    basis = int(round(np.mean([np.mean(iters_map[(l, 42)]) for l in LEAF_CONFIGS])))
    tr_c, (te_c,) = add_counts(feats_full, [feats_test])
    te_full = te_crossfit(tr_c, target, TE_COLS_ALL)
    te_test = te_apply(features, target, test_features, TE_COLS_ALL)
    X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
    X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
    X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
    preds = []
    for leaves in LEAF_CONFIGS:
        for seed in [42, 43, 44]:
            cfg = dict(learning_rate=0.05, num_leaves=leaves, n_estimators=basis,
                       n_jobs=-1, random_state=seed, verbose=-1)
            cfg.update(stoch)
            model = lgb.LGBMClassifier(**cfg)
            model.fit(X_full, target)
            preds.append(model.predict_proba(X_test)[:, 1])
    p = np.mean(preds, axis=0)
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': p})
    sub.to_csv('submission_round17.csv', index=False)
    print(f'submission_round17.csv written: {len(sub):,} rows | {tag}, 6 fits, basis {basis}. '
          f'Local file only - NOT submitted; Andrea submits.')


In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round17_oof.npz',
         incumbent=oof_inc, exp42=oof42, exp43=oof43, target=target.to_numpy())
meta = {
    'round': 17,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'exp42': {'scheme': 'Exp 36 with feature_fraction 0.8 (one parameter change)'},
        'exp43': {'scheme': 'Exp 36 + raw delay_total (NaN-preserving sum of the 2 delay columns); counts/TE on original columns only'},
        'delay_cols': DELAY_COLS,
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp42': aucs42, 'exp43': aucs43},
    'guard': {'winner': win, 'bar': 'E36_FOLDS_R12'},
}
(cache_dir / 'round17_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round17_oof.npz + round17_oof.json')


## What happens next

- If Exp 42 or Exp 43 wins (bounded to that configuration): submission_round17.csv becomes the
  new local candidate.
- If both null: these two configurations are measured without a detected increment, bounded to
  themselves - not evidence about other untested directions. Documented remaining directions
  on existing data narrow further, and each new test is another adaptive-selection bet on the
  same OOF. The state of campaign is reported plainly; external data (host-linked original
  dataset, CC0 on its face, provenance caveats logged) remains Andrea's explicit choice on its
  own merits.
